In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"), override=True)

%load_ext autoreload
%autoreload 2

## 上下文隔离：子智能体

<img src="./assets/agent_header_subagent.png" width="800" style="display:block; margin-left:0;">

随着对话的进行，智能体上下文可能会快速增长，导致几个与长上下文相关的问题。一个主要问题是上下文冲突或混乱，同一上下文窗口内的混合目标可能导致次优性能。[上下文隔离](https://blog.langchain.com/context-engineering-for-agents/)通过将任务委托给[专业化子智能体](https://www.anthropic.com/engineering/multi-agent-research-system)提供了有效的解决方案，每个子智能体都在自己的隔离上下文窗口内操作。这种方法防止上下文冲突、混乱、污染和稀释，同时实现专注的、专业化的任务执行。

### 子智能体委托
![./assets/subagents.png](./assets/subagents.png)
主要洞察是我们可以创建具有针对特定任务的不同工具集的子智能体。每个子智能体都存储在注册字典中，以 `subagent_type` 作为键，允许主智能体通过 `task(description, subagent_type)` 工具调用委托工作。子智能体与父级上下文完全隔离操作，其结果作为 `ToolMessage` 返回给父智能体，保持关注点的清晰分离。

## 步骤 1：创建子智能体

让我们定义用户将如何指定子智能体
```python
from typing_extensions import TypedDict

class SubAgent(TypedDict):
    """专业化子智能体的配置。"""

    name: str
    description: str
    prompt: str
    tools: NotRequired[list[str]]
```

我们将使用这些对象的列表来创建我们可以访问的所有子智能体

```python
agents: list[SubAgent] = ...
subagents = {
    agent['name']: create_react_agent(
        model=model,
        prompt=agent['prompt'],
        tools = get_tools(agent['tools']),
        ...
    )
}
```

## 步骤 2：创建使用子智能体的工具

从逻辑上讲，应该看起来像：

```python
def task(
    description: str  # 子智能体应该做的任务
    subagent_type: str  # 使用哪个子智能体
):
    # 创建新消息传递给子智能体 - 应该只是描述
    # 调用子智能体
    # 用子智能体的响应和对文件系统的任何更改更新状态
```

完整版本如下所示：

```python
@tool(description=TASK_DESCRIPTION_PREFIX.format(other_agents=other_agents_string))
def task(
    description: str,
    subagent_type: str,
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
):
    """将任务委托给具有隔离上下文的专业化子智能体。

    这为子智能体创建了一个仅包含任务描述的新上下文，
    防止来自父智能体对话历史的上下文污染。
    """
    # 验证请求的智能体类型是否存在
    if subagent_type not in agents:
        return f"错误：调用了类型为 {subagent_type} 的智能体，唯一允许的类型是 {[f'`{k}`' for k in agents]}"

    # 获取请求的子智能体
    sub_agent = agents[subagent_type]

    # 创建仅包含任务描述的隔离上下文
    # 这是上下文隔离的关键 - 没有父历史
    state["messages"] = [{"role": "user", "content": description}]

    # 在隔离中执行子智能体
    result = sub_agent.invoke(state)

    # 通过 Command 状态更新将结果返回给父智能体
    return Command(
        update={
            "files": result.get("files", {}),  # 合并任何文件更改
            "messages": [
                # 子智能体结果在父上下文中成为 ToolMessage
                ToolMessage(
                    result["messages"][-1].content, tool_call_id=tool_call_id
                )
            ],
        }
    )

return task
```

In [ ]:
%%writefile ../src/deep_agents_from_scratch/task_tool.py
"""通过子智能体进行上下文隔离的任务委托工具。

此模块提供了创建和管理具有隔离上下文的子智能体的核心基础设施。
子智能体通过在仅包含其特定任务描述的干净上下文窗口中操作来防止上下文冲突。
"""

from typing import Annotated, NotRequired
from typing_extensions import TypedDict

from langchain_core.messages import ToolMessage
from langchain_core.tools import BaseTool, InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState, create_react_agent
from langgraph.types import Command

from deep_agents_from_scratch.prompts import TASK_DESCRIPTION_PREFIX
from deep_agents_from_scratch.state import DeepAgentState


class SubAgent(TypedDict):
    """专业化子智能体的配置。"""

    name: str
    description: str
    prompt: str
    tools: NotRequired[list[str]]


def _create_task_tool(tools, subagents: list[SubAgent], model, state_schema):
    """创建一个任务委托工具，通过子智能体实现上下文隔离。

    此函数实现了生成具有隔离上下文的专业化子智能体的核心模式，
    防止复杂多步骤任务中的上下文冲突和混乱。

    Args:
        tools: 可分配给子智能体的可用工具列表
        subagents: 专业化子智能体配置列表
        model: 用于所有智能体的语言模型
        state_schema: 状态模式（通常是 DeepAgentState）

    Returns:
        一个可以将工作委托给专业化子智能体的 'task' 工具
    """
    # 创建智能体注册表
    agents = {}

    # 构建工具名称映射以进行选择性工具分配
    tools_by_name = {}
    for tool_ in tools:
        if not isinstance(tool_, BaseTool):
            tool_ = tool(tool_)
        tools_by_name[tool_.name] = tool_

    # 根据配置创建专业化子智能体
    for _agent in subagents:
        if "tools" in _agent:
            # 如果指定，使用特定工具
            _tools = [tools_by_name[t] for t in _agent["tools"]]
        else:
            # 默认使用所有工具
            _tools = tools
        agents[_agent["name"]] = create_react_agent(
            model, prompt=_agent["prompt"], tools=_tools, state_schema=state_schema
        )

    # 生成可用子智能体的描述以供工具描述使用
    other_agents_string = [
        f"- {_agent['name']}: {_agent['description']}" for _agent in subagents
    ]

    @tool(description=TASK_DESCRIPTION_PREFIX.format(other_agents=other_agents_string))
    def task(
        description: str,
        subagent_type: str,
        state: Annotated[DeepAgentState, InjectedState],
        tool_call_id: Annotated[str, InjectedToolCallId],
    ):
        """将任务委托给具有隔离上下文的专业化子智能体。

        这为子智能体创建了一个仅包含任务描述的新上下文，
        防止来自父智能体对话历史的上下文污染。
        """
        # 验证请求的智能体类型是否存在
        if subagent_type not in agents:
            return f"错误：调用了类型为 {subagent_type} 的智能体，唯一允许的类型是 {[f'`{k}`' for k in agents]}"

        # 获取请求的子智能体
        sub_agent = agents[subagent_type]

        # 创建仅包含任务描述的隔离上下文
        # 这是上下文隔离的关键 - 没有父历史
        state["messages"] = [{"role": "user", "content": description}]

        # 在隔离中执行子智能体
        result = sub_agent.invoke(state)

        # 通过 Command 状态更新将结果返回给父智能体
        return Command(
            update={
                "files": result.get("files", {}),  # 合并任何文件更改
                "messages": [
                    # 子智能体结果在父上下文中成为 ToolMessage
                    ToolMessage(
                        result["messages"][-1].content, tool_call_id=tool_call_id
                    )
                ],
            }
        )

    return task

现在，您有一个将生成子智能体作为工具的例程。现在，您可以定义特定的子智能体并允许系统使用 `task` 工具调用它们。
上面，`_create_task_tool` 接收一个类型为 `SubAgent` 的列表。此列表包含要创建的智能体的描述。

```python
class SubAgent(TypedDict):
    """专业化子智能体的配置。"""

    name: str
    description: str
    prompt: str
    tools: NotRequired[list[str]]


def _create_task_tool(tools, subagents: list[SubAgent], model, state_schema):
    """创建一个任务委托工具，通过子智能体实现上下文隔离。

```
`SubAgent` 类定义了满足子智能体双重角色所需的唯一信息。子智能体既充当工具又充当智能体。

- **作为工具**，它们向监督智能体提供有关其功能以及如何调用它们的信息。
- **作为智能体**，它们需要一个描述如何执行其任务的提示，以及一组针对这些任务的工具。

下面，您将创建一个研究子智能体。它的 `description` 告知监督智能体应该将单个任务委托给这个子智能体。`SIMPLE_RESEARCH_INSTRUCTIONS` 是子智能体用来指导其研究的提示。在此示例中，它很简短，但对于通用研究人员，它可能更加详细。子智能体还配备了一个 `web_search` 工具，用于研究期间使用。

```python
# 创建研究子智能体
research_sub_agent = {
    "name": "research-agent",
    "description": "将研究委托给子智能体研究员。一次只给这个研究员一个主题。",
    "prompt": SIMPLE_RESEARCH_INSTRUCTIONS,
    "tools": ["web_search"],
}
```

注意，子智能体接收特定任务以及完成任务所需的工具。它在自己的上下文中操作，仅限于单个任务描述。这种[上下文工程](https://blog.langchain.com/context-engineering-for-agents/)方法确保子智能体的工作上下文保持无上下文冲突、混乱、污染和稀释。

监督智能体提示现在必须包含如何调用和使用这些子智能体的描述。如下所示。注意*可用工具*描述和在适用时使用并行研究的说明。

In [ ]:
from utils import show_prompt

from deep_agents_from_scratch.prompts import SUBAGENT_USAGE_INSTRUCTIONS

show_prompt(SUBAGENT_USAGE_INSTRUCTIONS)

现在让我们构建一个带有监督者和子智能体的研究系统。这将只是一个带有预定义搜索结果的模拟版本，以演示各部分是如何组合在一起的。在下一课中，您将构建一个完整的研究系统。

In [ ]:
from datetime import datetime

from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

from deep_agents_from_scratch.prompts import SUBAGENT_USAGE_INSTRUCTIONS
from deep_agents_from_scratch.state import DeepAgentState
from deep_agents_from_scratch.task_tool import _create_task_tool

# 限制
max_concurrent_research_units = 3
max_researcher_iterations = 3

# 模拟搜索结果
search_result = """模型上下文协议（MCP）是由 Anthropic 开发的开放标准协议，
用于实现 AI 模型与外部系统（如工具、数据库和其他服务）之间的无缝集成。
它充当标准化通信层，允许 AI 模型以一致和高效的方式访问和利用来自各种来源的数据。
本质上，MCP 通过为数据交换提供统一语言来简化将 AI 助手连接到外部服务的过程。"""


# 模拟搜索工具
@tool(parse_docstring=True)
def web_search(
    query: str,
):
    """搜索网络以获取特定主题的信息。

    此工具执行网络搜索并返回给定查询的相关结果。
    当您需要从互联网收集有关任何主题的信息时使用此工具。

    Args:
        query: 搜索查询字符串。要具体明确您要查找的信息。

    Returns:
        来自搜索引擎的搜索结果。

    Example:
        web_search("机器学习在医疗保健中的应用")
    """
    return search_result


# 添加模拟研究指令
SIMPLE_RESEARCH_INSTRUCTIONS = """您是一名研究员。研究提供给您的主题。重要：只需调用一次 web_search 工具，并使用工具提供的结果来回答提供的主题。"""

# 创建研究子智能体
research_sub_agent = {
    "name": "research-agent",
    "description": "将研究委托给子智能体研究员。一次只给这个研究员一个主题。",
    "prompt": SIMPLE_RESEARCH_INSTRUCTIONS,
    "tools": ["web_search"],
}

# 直接使用 create_react_agent 创建智能体
model = init_chat_model(model="anthropic:claude-sonnet-4-20250514", temperature=0.0)

# 子智能体工具
sub_agent_tools = [web_search]

# 创建任务工具以将任务委托给子智能体
task_tool = _create_task_tool(
    sub_agent_tools, [research_sub_agent], model, DeepAgentState
)

# 工具
delegation_tools = [task_tool]

# 使用系统提示创建智能体
agent = create_react_agent(
    model,
    delegation_tools,
    prompt=SUBAGENT_USAGE_INSTRUCTIONS.format(
        max_concurrent_research_units=max_concurrent_research_units,
        max_researcher_iterations=max_researcher_iterations,
        date=datetime.now().strftime("%a %b %-d, %Y"),
    ),
    state_schema=DeepAgentState,
)

# 显示智能体
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
from utils import format_messages

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "给我一个模型上下文协议（MCP）的概述。",
            }
        ],
    }
)

format_messages(result["messages"])

跟踪：
https://smith.langchain.com/public/26cc1c2b-e785-4c6d-a2a7-c30a31875fc7/r
<!-- https://smith.langchain.com/public/edc4e672-db9c-457a-953d-f62e7813591c/r -->